In [2]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import os
import random  # ← AGGIUNTO!
from datetime import datetime

print("🌐 PIPELINE ETL NASA WEB LOGS (NO Download!)")

# 1. GENERA DATASET NASA (10k log realistici)
print("📥 Creazione dataset NASA...")
data = []
for i in range(10000):
    ora = f"{random.randint(0,23):02d}:{random.randint(0,59):02d}:{random.randint(0,59):02d}"
    data.append({
        'timestamp': f'29/Apr/2026:{ora} +0200',
        'ip': f"{random.randint(1,255)}.{random.randint(1,255)}.{random.randint(1,255)}.{random.randint(1,255)}",
        'method': random.choice(['GET', 'POST']),
        'url': random.choice(['/home', '/images/logo.gif', '/search', '/shop/product1', '/api/user']),
        'status': random.choice([200, 404, 500, 301, 200, 200]),
        'bytes': random.randint(100, 25000),
        'user_agent': random.choice(['Mozilla/Chrome', 'Bot-Crawler', 'Safari-iPhone', 'Firefox'])
    })

df_logs = pd.DataFrame(data)
print(f"✅ {len(df_logs):,} log NASA creati!")

# 2. BRONZE LAYER
os.makedirs('bronze_nasa', exist_ok=True)
df_logs.to_csv('bronze_nasa/raw_nasa_logs.csv', index=False)

# 3. SILVER ENRICHMENT
print("🔄 Arricchimento dati...")
df_silver = df_logs.copy()
df_silver['hour'] = df_silver['timestamp'].str.extract('(\d{2})').astype(int)[0]
df_silver['error'] = df_silver['status'] >= 400
df_silver['page_type'] = df_silver['url'].apply(
    lambda x: 'home' if 'home' in x else 'image' if 'image' in x else 'api' if 'api' in x else 'other'
)
os.makedirs('silver_nasa', exist_ok=True)
df_silver.to_csv('silver_nasa/enriched_logs.csv', index=False)

# 4. DASHBOARD AZIENDALE
print("📈 Creazione dashboard...")
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('NASA Web Server Analytics Dashboard', fontsize=16)

# Top pagine
df_silver['page_type'].value_counts().plot.bar(ax=axes[0,0], color='purple')
axes[0,0].set_title('🏆 Tipi di Pagina')

# Traffico orario
df_silver['hour'].value_counts().sort_index().plot.line(ax=axes[0,1], marker='o', color='blue')
axes[0,1].set_title('⏰ Picchi Orari')

# Errori
df_silver['error'].value_counts().plot.pie(ax=axes[1,0], autopct='%1.1f%%')
axes[1,0].set_title('⚠️ Tasso Errori')

# Status codes
df_silver['status'].astype(int).value_counts().head(8).plot.bar(ax=axes[1,1], color='red')
axes[1,1].set_title('📊 HTTP Status')

plt.tight_layout()
plt.savefig('nasa_dashboard.png', dpi=300, bbox_inches='tight')
plt.show()

# 5. DATA WAREHOUSE + INSIGHTS
print("🏢 Data Warehouse...")
conn = sqlite3.connect('nasa_web_dw.db')
df_silver.to_sql('nasa_logs', conn, if_exists='replace', index=False)

insights = pd.read_sql("""
    SELECT page_type, 
           COUNT(*) as richieste,
           ROUND(AVG(bytes), 0) as bandwidth_medio,
           SUM(error) as errori,
           ROUND(100.0*SUM(error)/COUNT(*), 2) as error_rate_pct
    FROM nasa_logs 
    GROUP BY page_type 
    ORDER BY richieste DESC
""", conn)

insights.to_csv('nasa_business_insights.csv', index=False)
print("\n💼 INSIGHTS PER AZIENDE:")
print(insights)

conn.close()

print("\n🎉 === PIPELINE NASA COMPLETATA! ===")
print("File generati per GitHub:")
print("- bronze_nasa/raw_nasa_logs.csv")
print("- silver_nasa/enriched_logs.csv")
print("- nasa_web_dw.db")
print("- nasa_dashboard.png")
print("- nasa_business_insights.csv")
print("\n💼 Repo: nasa-web-logs-etl")

🌐 PIPELINE ETL NASA WEB LOGS (Dataset Incluso!)
📥 Generazione dataset NASA logs...


NameError: name 'random' is not defined